# <img style="float: left; padding-right: 10px; width: 100px" src="https://raw.githubusercontent.com/sbildea/SEED/main/imagini/unitbv_mie.png"> <center>Știința Explorării și Exploatării Datelor - SEED1</center>


##    <center>Tema 4: Regresie: prețul mașinilor second-hand</center>
**Universitatea Transilvania**<br>
**SIIPA, Semestrul I**<br>
**Instructor:** Conf. dr. Teodor &Scedil;tefan B&icirc;ldea<br>
**Contact:** teodor.bildea@unitbv.ro<br>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sbildea/SEED/blob/main/cursuri/04-regresie/tema-04-regresie.ipynb)

In [ ]:
# Configurare curs SEED: rulați această celulă prima.
# Colab: descarcă materialele cursului și instalează pachetele lipsă (aprox. un minut).
# Local: găsește singură rădăcina repository-ului, oricare ar fi folderul curent al kernelului.
import os, sys
MODUL = "04-regresie"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/SEED"):
        !git clone -q --depth 1 https://github.com/sbildea/SEED.git /content/SEED
        !pip install -q -r /content/SEED/requirements.txt
    RADACINA = "/content/SEED"
else:
    RADACINA = os.path.abspath(".")
    while not os.path.isdir(os.path.join(RADACINA, "seed")) and os.path.dirname(RADACINA) != RADACINA:
        RADACINA = os.path.dirname(RADACINA)
    import importlib.util, socket
    lipsa = [p for p in ("pandas", "matplotlib", "seaborn", "sklearn") if importlib.util.find_spec(p) is None]
    if lipsa:
        print("Lipsesc pachetele", lipsa, "- în terminal: pip install -r requirements-local.txt")
    try:
        socket.create_connection(("raw.githubusercontent.com", 443), 2).close()
    except OSError:
        print("Fără internet: imaginile din text nu se văd. O singură dată, în terminal: python tools/imagini_offline.py")
os.chdir(os.path.join(RADACINA, "cursuri", MODUL))
sys.path.insert(0, RADACINA)

## Cum lucrezi la această temă

1. Deschide notebook-ul:
   - în **Colab** (butonul **Open in Colab**) și salvează-ți o copie: **File → Save a copy in Drive**; vezi [ghidul Colab](https://github.com/sbildea/SEED/blob/main/docs/ghid-colab.md);
   - sau **local**, în VS Code, pe o copie a fișierului (merge și fără internet); vezi [ghidul local](https://github.com/sbildea/SEED/blob/main/docs/ghid-local.md).
2. Rulează celula **Configurare curs** de mai sus. Datele se încarcă apoi cu o singură linie, de exemplu `incarca("date-curs", "audi")`.
3. Pentru fiecare exercițiu:
   - citește conceptul din curs și formulează **în cuvintele tale** ce vrei să obții și de ce;
   - cere codul unui asistent AI (Gemini din Colab, Claude, ChatGPT etc.) printr-o cerere în limbaj natural; vezi [ghidul de programare asistată de AI](https://github.com/sbildea/SEED/blob/main/docs/ghid-programare-asistata-ai.md);
   - rulează codul și **verifică rezultatul**: are sens? se potrivește cu datele? Corectează cererea dacă e nevoie;
   - notează în **Jurnalul AI** de la finalul notebook-ului cererile importante și ce ai verificat.
4. Redenumește fișierul conform indicațiilor din Prezentarea generală și predă-l.

> Evaluarea pune accentul pe înțelegerea conceptelor: trebuie să poți explica fiecare rezultat și de ce ai ales metoda respectivă.

> Întrebări: teodor.bildea@unitbv.ro

## Etica universitară

### Colaborare vs plagiat

Colaborarea între voi este încurajată atât timp cât nu cade în categoria plagiatului așa cum este descris în statutul studentului.

Copiatul/pagiatul și orice altă formă de incorectitudine academică nu vor fi tolerate și vor fi sancționate conform statutului studentului.

## Datele și regulile temei

Un dealer de mașini second-hand: *„Cât să cerem pe un Audi la mâna a doua și cât de mult ne putem înșela?”*

- Datele: tabelul `audi` din setul `date-curs` (anunțuri de mașini second-hand din Marea Britanie; prețul `price` este în **lire sterline**, £).
- La fiecare împărțire train/test folosiți **20% test** și `random_state` = **numărul vostru de ordine din catalog**.
- Lucrați ca la curs (`curs-04a`, `curs-04b`): modelul de referință, pregătirea în `Pipeline`, evaluarea pe test. Lipiți asistentului primerul de la începutul `curs-04a` și spuneți-i că lucrați pe alt set de date.

# Exercițiul 1. Pregătirea și ștacheta

- Încărcați datele și uitați-vă la ele: tipurile coloanelor, valori lipsă, **rânduri duplicate**, valorile coloanei `model` (cum sunt scrise exact?), valori imposibile (de exemplu un motor de 0 litri). Decideți ce faceți cu fiecare problemă găsită și explicați de ce.
- Împărțiți datele în antrenare și test.
- Calculați MAE, RMSE și R² pe test pentru **modelul de referință** (media prețurilor din antrenare).

*Întrebare:* cu câte lire greșește, în medie, o estimare „după medie”?

In [ ]:
# codul vostru aici
from seed.date import incarca
auto = incarca("date-curs", "audi")
auto.head()

# Exercițiul 2. O dreaptă

- Antrenați o regresie liniară simplă `price` ~ `mileage` (kilometrajul, în mile) și una `price` ~ `year`. Pentru fiecare: panta, interceptul, MAE și R² pe antrenare și pe test, și un grafic de împrăștiere cu dreapta.

*Întrebări:* cu câte lire scade prețul estimat la fiecare 10.000 de mile în plus? Care dintre cele două coloane explică mai mult din preț? Există mașini care se abat mult de la dreaptă și ce au în comun?

In [ ]:
# codul vostru aici

# Exercițiul 3. Toate coloanele și regularizarea

- Construiți un `Pipeline` care codifică one-hot coloanele text (`model`, `transmission`, `fuelType`) și antrenați: regresia liniară, `RidgeCV`, `LassoCV` și `ElasticNetCV` (pentru ultimele trei, cu standardizare și o listă de valori `alpha` care merge de la 0,01 la 10.000).
- Puneți toate modelele, plus modelul de referință, într-un singur tabel cu MAE și R² pe antrenare și pe test.
- Pentru fiecare model regularizat: ce `alpha` a fost ales? Câți coeficienți anulează Lasso?

*Întrebări:* aduce regularizarea vreun câștig aici? Comparați cu cazul celor 150 de case din `curs-04b` (capitolul 6) și explicați diferența gândindu-vă la numărul de rânduri și de coloane. Există modele de mașină reprezentate de foarte puține anunțuri? Ce înseamnă asta pentru estimările lor?

In [ ]:
# codul vostru aici

# Exercițiul 4. Ce îi spunem dealerului

Scrieți **3 propoziții** pentru dealer, cu cifre calculate în notebook-ul vostru:
1. cât greșește, în medie, modelul ales, față de o estimare după medie;
2. sub ce eroare se încadrează 8 mașini din 10 (cuantila 80% a erorilor absolute pe test);
3. o limită a modelului (pentru ce mașini nu v-ați baza pe el?).

*Răspunsul meu:*

In [ ]:
# codul vostru aici (dacă e nevoie)

# Jurnal AI

Completați pe scurt, pentru fiecare exercițiu, cum ați lucrat cu asistentul AI.

| Exercițiu | Cererea principală (prompt) | Ce am verificat / corectat | Ce am învățat |
|---|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |
| 4 |  |  |  |